# ПЗ 3 — OCR из кадров (субтитры)

**Задача:** читать кадры из Drive (результат ПЗ 2), распознать субтитры через EasyOCR, сохранить CSV в Drive.

**Стек:** `easyocr`, `opencv-python`, `pandas`

In [ ]:
!pip install easyocr opencv-python-headless pandas tqdm -q

In [ ]:
from google.colab import drive
import os

# === МОНТИРОВАНИЕ DRIVE ===
drive.mount('/content/drive', force_remount=True)

# === ПАПКИ (те же что в ПЗ 2) ===
BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Проверка кадров
if not os.path.exists(FRAMES_DIR):
    print('❌ Папка с кадрами не найдена — сначала запустите ПЗ 2')
else:
    frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
    print(f'✅ Найдено кадров: {len(frames)}')

In [ ]:
import cv2
import easyocr
import pandas as pd
from tqdm.notebook import tqdm

SUBTITLE_FRACTION = 0.15  # нижние 15% кадра — зона субтитров
reader = easyocr.Reader(['ru', 'en'], gpu=True)
print('✅ EasyOCR готов')

In [ ]:
results = []

for fname in tqdm(frames, desc='OCR'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue
    h = img.shape[0]
    crop = img[int(h * (1 - SUBTITLE_FRACTION)):, :]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    for _, text, prob in reader.readtext(gray):
        text = text.strip()
        if text and prob > 0.4:
            results.append({'frame': fname, 'text': text, 'conf': round(prob, 3)})

df = pd.DataFrame(results)
OUT = f'{RESULTS_DIR}/ocr_results.csv'
df.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f'✅ Распознано строк: {len(df)} → {OUT}')

In [ ]:
# Предварительная дедупликация
unique_texts = df['text'].drop_duplicates().reset_index(drop=True)
print(f'Уникальных фраз: {len(unique_texts)}')
print(unique_texts.head(20).to_string())